# JengaNLP quickstart: train a Swahili / Sheng text classifier in your browser

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rogendo/JengaNLP/blob/main/notebooks/jenganlp_quickstart.ipynb)

[JengaNLP](https://github.com/Rogendo/JengaNLP) trains transformer models from a single YAML file:
one shared encoder, any number of tasks and heads, then load-once inference.

In this notebook you will:

1. install `jenga-nlp` from PyPI,
2. write a small dataset of M-PESA messages in English, Swahili and Sheng,
3. describe the experiment in YAML,
4. train, evaluate and predict from the command line,
5. load the model once in Python and score many messages,
6. train **one** model that does two tasks (fraud detection and sentiment).

**Runtime:** choose *Runtime → Change runtime type → T4 GPU* in Colab; the whole notebook then takes
about 5 minutes. It also runs on CPU, but each training step takes 15–25 minutes there.

> The dataset is tiny and made up, so the model learns the workflow, not production-quality
> fraud detection. Swap in your own data once the steps make sense.

Docs: https://rogendo.github.io/JengaNLP/

## 1. Install JengaNLP

In [ ]:
%pip install -q jenga-nlp

In [ ]:
import jenga_nlp, torch

print("jenga-nlp", jenga_nlp.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU is fine)")

The CLI lists its commands (`train`, `evaluate`, `predict`, `resume`, ...):

In [ ]:
!jenga-nlp --help

## 2. Create a dataset

JengaNLP reads JSONL: one JSON object per line with a text column and a label column.
Labels are integer class ids; `label_maps` in the config gives them readable names.
Identifiers such as names are written as placeholders (`[NAME]`, `[LOCATION]`).

In [ ]:
import json, os

WORK = "jenga_quickstart"
os.makedirs(WORK, exist_ok=True)

fraud_rows = [
    ("Your M-PESA account will be suspended today. Send your PIN to 0700 000 000 to keep it active.", "fraud"),
    ("Habari, nimekutumia KSh 2,000 kimakosa. Tafadhali nirudishie kwa namba hii haraka.", "fraud"),
    ("Congratulations! You have won KSh 50,000 in the Safari promo. Pay KSh 500 registration to claim.", "fraud"),
    ("Msee, ni agent wa Safaricom. Nipe ile code umetumiwa kwa SMS nikufungulie line.", "fraud"),
    ("Akaunti yako imefungwa. Tuma namba yako ya siri na ID ili tuifungue.", "fraud"),
    ("URGENT: your loan limit has been raised to KSh 80,000. Click the link and enter your PIN to activate.", "fraud"),
    ("Mamii, niko na shida hospitali, nitumie elfu tano kwa hii namba mpya, simu yangu imeibiwa.", "fraud"),
    ("Dear customer, a reversal of KSh 3,400 is pending. Reply with your M-PESA PIN to approve.", "fraud"),
    ("Umechaguliwa kupata kazi Dubai. Lipa KSh 15,000 ya visa leo ili tukuhifadhie nafasi.", "fraud"),
    ("Bro, nimekosea nikakutumia doh, ni ya rent. Nirudishie fasta tafadhali, nitakubless.", "fraud"),
    ("Your Fuliza is blocked. Call this number and give the officer your PIN and ID number to unblock.", "fraud"),
    ("Pata mkopo wa KSh 100,000 bila guarantor. Tuma KSh 1,000 ya processing kwanza.", "fraud"),
    ("Hii ni KRA. Una deni la ushuru, lipa sasa hivi kwa till hii la sivyo utakamatwa.", "fraud"),
    ("Investment ya crypto inadouble pesa ndani ya wiki moja. Tuma elfu kumi uanze leo.", "fraud"),
    ("Safaricom: we are upgrading your SIM. Switch off your phone for 3 hours and share the code we send.", "fraud"),
    ("Ndugu, ni mimi pastor. Kanisa linahitaji mchango wa haraka, tuma kwa hii namba binafsi.", "fraud"),
    ("Confirmed. You have received KSh 6,500 from [NAME] on 12/3 at 9:14 AM. New M-PESA balance is KSh 7,120.", "legit"),
    ("Umelipa KSh 1,200 kwa KPLC PREPAID. Salio lako jipya la M-PESA ni KSh 3,450.", "legit"),
    ("Mum, nimefika shule salama. Asante kwa fare, nitakupigia jioni.", "legit"),
    ("Your Fuliza M-PESA limit is KSh 2,000. Dial *234# to check your balance.", "legit"),
    ("Confirmed. KSh 350 paid to [NAME] matatu sacco. Transaction cost KSh 0.", "legit"),
    ("Msee nimekutumia ile 500 ya lunch, check M-PESA yako.", "legit"),
    ("Malipo yako ya karo ya KSh 18,000 yamepokelewa na shule. Asante.", "legit"),
    ("Reminder: Safaricom will never ask for your M-PESA PIN. Report fraud calls to 333.", "legit"),
    ("Nimelipa bill ya maji kwa paybill, ukipata message niambie.", "legit"),
    ("You have bought KSh 100 airtime for your number. M-PESA balance is KSh 980.", "legit"),
    ("Chama meeting ni Jumamosi saa nane. Kila mtu alete mchango wa mwezi huu.", "legit"),
    ("Confirmed. KSh 2,500 withdrawn from agent [NAME] at [LOCATION]. Fee KSh 29.", "legit"),
    ("Boss, nimetuma doh ya stock. Ukipata nitumie risiti tafadhali.", "legit"),
    ("Your M-Shwari savings earned KSh 12 interest this month.", "legit"),
    ("Kesho nitakutumia pesa ya shamba baada ya kuuza maziwa.", "legit"),
    ("Thank you for paying KSh 4,000 to [NAME] Hardware. Your receipt number is in the app.", "legit"),
    ("Wewe ni mshindi wa droo ya Lipa na M-PESA. Tuma KSh 250 ya ushuru kupokea zawadi yako.", "fraud"),
    ("Hello, I am calling from the bank. Read me the OTP we just sent so we can secure your account.", "fraud"),
    ("Sasa msee, nimekutumia token ya KPLC kimakosa. Tafadhali nitumie pesa yake kwa namba hii.", "fraud"),
    ("Your SIM card will be deactivated in 24 hours. Send your ID number and date of birth to update.", "fraud"),
    ("Tuma KSh 2,000 sasa na upokee KSh 20,000 kesho. Ofa ni ya leo tu.", "fraud"),
    ("Mtoto wako amepata ajali, tuma pesa ya hospitali kwa daktari huyu haraka kabla ya upasuaji.", "fraud"),
    ("Final notice: your HELB account is in default. Pay the clearance fee to this personal number today.", "fraud"),
    ("Nimeona CV yako. Kazi ya supermarket iko, lipa KSh 3,000 ya medical kwanza.", "fraud"),
    ("Your parcel is held at customs. Pay KSh 1,850 release fee via this till to receive it.", "fraud"),
    ("Mkopo wako wa KSh 45,000 umekubaliwa. Tuma PIN yako ili tuuweke kwenye M-PESA.", "fraud"),
    ("Bro, ni mimi, nimebadilisha line. Nitumie 3k fasta, nitakurudishia jioni.", "fraud"),
    ("Akaunti yako ya M-Shwari ina tatizo. Bonyeza link hii na uweke PIN kuthibitisha.", "fraud"),
    ("We have detected suspicious login on your account. Share the verification code to block it.", "fraud"),
    ("Pata uraia wa Canada kwa wiki mbili tu. Tuma KSh 20,000 ya application leo.", "fraud"),
    ("Sorry boss, nimekutumia 7,500 kwa makosa badala ya mama mboga. Tafadhali rudisha.", "fraud"),
    ("Safaricom bonus: dial this code and enter your PIN to receive 5GB free data.", "fraud"),
    ("Confirmed. KSh 1,500 sent to [NAME] for school shoes. New M-PESA balance is KSh 4,210.", "legit"),
    ("Habari mteja, bili yako ya Zuku ya mwezi huu ni KSh 2,999. Lipa kupitia paybill kabla ya tarehe 5.", "legit"),
    ("Sasa, nimefika stage. Nitachukua matatu ya saa tatu, tutaonana town.", "legit"),
    ("Your NHIF contribution of KSh 500 has been received. Thank you.", "legit"),
    ("Confirmed. You bought 1GB data bundle for KSh 99. Valid for 24 hours.", "legit"),
    ("Mama, nimepokea ile pesa ya mbolea. Nitanunua kesho asubuhi.", "legit"),
    ("Your Fuliza loan of KSh 300 has been fully repaid. Thank you for using M-PESA.", "legit"),
    ("Kikao cha wazazi ni Ijumaa saa nne asubuhi shuleni. Karibuni wote.", "legit"),
    ("Confirmed. KSh 650 paid to [NAME] Butchery. Transaction cost KSh 0.", "legit"),
    ("Niko kwa jam Thika Road, nitachelewa kama dakika ishirini.", "legit"),
    ("Your KPLC token for KSh 1,000 is ready. Units: 41.2 kWh.", "legit"),
    ("Buda, nimekutumia ile 2k ya mchango wa harusi. Ukipata niambie.", "legit"),
    ("Thank you for shopping at [NAME] Supermarket. You paid KSh 3,420 via M-PESA.", "legit"),
    ("Ombi lako la statement limepokelewa. Utaipata kwa email ndani ya saa 24.", "legit"),
    ("Confirmed. KSh 10,000 deposited to your account at agent [NAME], [LOCATION].", "legit"),
    ("Jamaa, mechi ni saa kumi na moja. Leta jezi zako.", "legit"),
]

FRAUD_LABELS = {"legit": 0, "fraud": 1}
with open(f"{WORK}/fraud.jsonl", "w", encoding="utf-8") as f:
    for text, label in fraud_rows:
        f.write(json.dumps({"text": text, "label": FRAUD_LABELS[label]}, ensure_ascii=False) + "\n")

print(len(fraud_rows), "examples written to", f"{WORK}/fraud.jsonl")

In [ ]:
!head -3 jenga_quickstart/fraud.jsonl

## 3. Describe the experiment in YAML

The whole experiment lives in one file:

- `model.base_model`: any Hugging Face encoder. [`Davlan/afro-xlmr-base`](https://huggingface.co/Davlan/afro-xlmr-base) is XLM-R adapted to 17 African languages, including Swahili. (The smaller `afro-xlmr-mini` was unstable on a dataset this small.)
- `tasks`: a list. Each task has a type, a data file and one or more heads.
- `training`: hyper-parameters and where the model is saved. `device: auto` uses the GPU when there is one.

Every option is documented in the [configuration reference](https://rogendo.github.io/JengaNLP/configuration/).

In [ ]:
%%writefile jenga_quickstart/fraud.yaml
project_name: quickstart_mpesa_fraud

model:
  base_model: Davlan/afro-xlmr-base
  dropout: 0.1

tokenizer:
  max_length: 64
  padding: max_length
  truncation: true

tasks:
  - name: mpesa_fraud
    type: single_label_classification
    data_path: fraud.jsonl          # relative to this YAML file
    text_column: text
    label_column: label
    heads:
      - name: fraud_head
        num_labels: 2
    label_maps:
      fraud_head: {0: legit, 1: fraud}

training:
  output_dir: jenga_quickstart/fraud_model
  learning_rate: 0.00003
  batch_size: 8
  eval_batch_size: 8
  num_epochs: 6
  warmup_steps: 6
  device: auto
  metric_for_best_model: eval_loss
  greater_is_better: false
  logging:
    service: none                  # set to mlflow or tensorboard to track runs
    experiment_name: quickstart_mpesa_fraud
  checkpoint:
    save_best: true
    max_checkpoints: 1
  data:
    test_size: 0.25                # 25% held out for evaluation
    seed: 42

## 4. Train

One command: it reads the YAML, downloads the base model, tokenizes the data, trains, evaluates
on the held-out split after each epoch and saves the best model to `training.output_dir`.

In [ ]:
!jenga-nlp train --config jenga_quickstart/fraud.yaml

## 5. Evaluate

Reload the saved model and report metrics (accuracy, precision, recall, F1) on the held-out split.
With 16 test examples one message changes accuracy by 6 points, so expect some variation between runs.

In [ ]:
!jenga-nlp evaluate --config jenga_quickstart/fraud.yaml --model-dir jenga_quickstart/fraud_model

## 6. Predict from the command line

Good for a quick check. It reloads the model on every call, so use Python (next step) for many texts.

In [ ]:
!jenga-nlp predict --config jenga_quickstart/fraud.yaml --model-dir jenga_quickstart/fraud_model \
    --text "Tuma PIN yako sasa hivi ili tuzuie akaunti yako isifungwe"

## 7. Load once in Python, predict many

`InferencePipeline` loads the model a single time. Use this in notebooks, batch jobs and web services.

In [ ]:
from jenga_nlp.inference.pipeline import InferencePipeline

pipe = InferencePipeline.from_checkpoint(
    model_dir="jenga_quickstart/fraud_model",
    config_path="jenga_quickstart/fraud.yaml",
)

messages = [
    "Confirmed. KSh 800 paid to [NAME] for groceries. New balance KSh 2,140.",
    "Niaje, ni customer care. Nitumie ile code ya SMS nikurudishie pesa yako.",
    "You have won a new phone! Send KSh 300 delivery fee to claim it today.",
    "Mum, nimelipa fees ya term hii. Risiti nitakuletea weekend.",
]

for result in pipe.predict_batch(messages):
    head = result.task_results["mpesa_fraud"].heads["fraud_head"]
    print(f"{head.label or head.prediction!s:>6}  {head.confidence:.2f}  {result.text}")

Every result is a structured object; `to_dict()` gives JSON you can return from an API:

In [ ]:
pipe.predict("Akaunti yako imefungwa, piga simu hii upesi").to_dict()

### Try your own message

In [ ]:
text = "Bro, nimekutumia doh kimakosa, nirudishie tafadhali"  # @param {type:"string"}
head = pipe.predict(text).task_results["mpesa_fraud"].heads["fraud_head"]
print(head.label or head.prediction, f"(confidence {head.confidence:.2f})")

## 8. One model, two tasks, with attention fusion

JengaNLP's main feature is multi-task learning: one shared encoder with a head per task, trained together.
Add a sentiment dataset and list both tasks in the same YAML.

`model.fusion.type: attention` adds a small layer between the encoder and the heads. Each task learns
an embedding that attends over the sentence's tokens; the result is added back through a learned gate,
so the fraud head and the sentiment head each see features tuned to their task instead of
identical ones. The layer starts as a no-op and learns how much task conditioning to add.
Use `type: concatenation` for a simpler variant, or remove the `fusion` block to share features as-is.

`training.task_sampling: round_robin` alternates batches between the tasks (`proportional` and
`temperature` weight them by dataset size).

In [ ]:
sentiment_rows = [
    ("M-PESA imeenda haraka sana leo, asante Safaricom!", "positive"),
    ("Customer care walinisaidia vizuri, tatizo limeisha.", "positive"),
    ("Hii app mpya iko fiti sana, inarahisisha kila kitu.", "positive"),
    ("Thank you, the reversal was done within minutes.", "positive"),
    ("Nimefurahi sana, mkopo umeingia bila stress.", "positive"),
    ("Service ya leo imekuwa poa, sina malalamiko.", "positive"),
    ("Network imekuwa mbaya wiki nzima, nimechoka.", "negative"),
    ("I waited two hours on the line and nobody helped me.", "negative"),
    ("Hii charges ni mob sana, siwezi endelea hivi.", "negative"),
    ("Pesa yangu imekatwa mara mbili na hakuna anayejibu.", "negative"),
    ("Agent alinijibu vibaya, sitarudi pale.", "negative"),
    ("The app keeps crashing every time I try to pay.", "negative"),
    ("Nataka kujua salio langu la M-PESA.", "neutral"),
    ("Office yenu inafunguliwa saa ngapi Jumamosi?", "neutral"),
    ("I would like to change the name on my account.", "neutral"),
    ("Nimetuma ombi la statement ya mwezi uliopita.", "neutral"),
    ("Paybill number ya KPLC ni gani?", "neutral"),
    ("Please send me the form for registering a business till.", "neutral"),
    ("Wow, the new Fuliza limit imenisaidia sana mwezi huu.", "positive"),
    ("Asanteni, agent wa hapa ni mpole na mwaminifu.", "positive"),
    ("The upgrade made payments so much faster, great job.", "positive"),
    ("Nimepata huduma nzuri sana kwa branch ya leo.", "positive"),
    ("Mbona mnakata pesa bila kuniarifu? Hii ni wizi.", "negative"),
    ("Hakuna network tangu asubuhi, biashara yangu imesimama.", "negative"),
    ("I am very disappointed, my reversal has taken a week.", "negative"),
    ("Hii app inakataa kufunguka, nimechoka kujaribu.", "negative"),
    ("Ningependa kubadilisha PIN yangu, nifanyeje?", "neutral"),
    ("What are the charges for sending KSh 5,000?", "neutral"),
    ("Branch ya Kisumu iko wapi?", "neutral"),
    ("Nataka kusajili line mpya kwa jina la mtoto wangu.", "neutral"),
]

SENTIMENT_LABELS = {"negative": 0, "neutral": 1, "positive": 2}
with open(f"{WORK}/sentiment.jsonl", "w", encoding="utf-8") as f:
    for text, label in sentiment_rows:
        f.write(json.dumps({"text": text, "label": SENTIMENT_LABELS[label]}, ensure_ascii=False) + "\n")
print(len(sentiment_rows), "sentiment examples")

In [ ]:
%%writefile jenga_quickstart/multitask.yaml
project_name: quickstart_multitask

model:
  base_model: Davlan/afro-xlmr-base
  fusion:
    type: attention                # task-conditioned features for each head
    num_attention_heads: 1

tokenizer:
  max_length: 64
  padding: max_length
  truncation: true

tasks:
  - name: mpesa_fraud
    type: single_label_classification
    data_path: fraud.jsonl
    label_column: label
    heads:
      - name: fraud_head
        num_labels: 2
    label_maps:
      fraud_head: {0: legit, 1: fraud}
  - name: customer_sentiment
    type: sentiment
    data_path: sentiment.jsonl
    label_column: label
    heads:
      - name: sentiment_head
        num_labels: 3
    label_maps:
      sentiment_head: {0: negative, 1: neutral, 2: positive}

training:
  output_dir: jenga_quickstart/multitask_model
  task_sampling: round_robin         # alternate batches between the tasks
  learning_rate: 0.00003
  batch_size: 8
  eval_batch_size: 8
  num_epochs: 6
  warmup_steps: 6
  device: auto
  logging:
    service: none
    experiment_name: quickstart_multitask
  checkpoint:
    save_best: true
    max_checkpoints: 1
  data:
    test_size: 0.25
    seed: 42

In [ ]:
!jenga-nlp train --config jenga_quickstart/multitask.yaml

Without `task_name`, `predict` runs every task on the text:

In [ ]:
multi = InferencePipeline.from_checkpoint(
    model_dir="jenga_quickstart/multitask_model",
    config_path="jenga_quickstart/multitask.yaml",
)

for text in [
    "Customer care wamenisaidia poa sana, asanteni!",
    "Tuma PIN yako ili tukurudishie pesa uliyotuma kimakosa.",
]:
    result = multi.predict(text)
    print(text)
    for task, tr in result.task_results.items():
        for name, head in tr.heads.items():
            print(f"   {task:>18}: {head.label or head.prediction} ({head.confidence:.2f})")

## 9. Optional: share your model on the Hugging Face Hub

The output folder holds the weights, tokenizer and the exact config, so anyone can reload it.
Log in with a token that has *write* access, then upload:

In [ ]:
# from huggingface_hub import notebook_login, upload_folder
# notebook_login()
# upload_folder(folder_path="jenga_quickstart/fraud_model",
#               repo_id="<your-username>/jenga-mpesa-fraud-demo", repo_type="model")

## Next steps

- Use your own data: point `data_path` at your JSONL file and set `num_labels` and `label_maps`.
- Other task types: [NER](https://rogendo.github.io/JengaNLP/tasks/ner/), [multi-label](https://rogendo.github.io/JengaNLP/tasks/multi-label/), [regression](https://rogendo.github.io/JengaNLP/tasks/regression/), [call-centre QA scoring](https://rogendo.github.io/JengaNLP/tasks/qa-scoring/).
- [Multi-task learning](https://rogendo.github.io/JengaNLP/tasks/multi-task/): task sampling strategies and per-head loss weights.
- [Production inference](https://rogendo.github.io/JengaNLP/inference/): batching, PII flags and a FastAPI wrapper.
- Track experiments: set `training.logging.service` to `mlflow` or `tensorboard`.

Questions or bugs: [open an issue](https://github.com/Rogendo/JengaNLP/issues).